# PlantScope v4 — Houseplant disease pipeline (training notebook)

Двухступенчатый пайплайн под комнатные растения. Все правки v4 нацелены на две регрессии прошлой версии: «всё больное» и «иногда не распознаёт растение».

| Стадия | Модель | Параметры | Что делает |
|--------|--------|-----------|------------|
| **Stage 1 — Detector** | YOLOv8n | ~3.2M | Локализация листьев (один класс `leaf`) |
| **Stage 2 — Classifier** | EfficientNet-B0 | ~5.3M | До 8 классов состояния листа + rejection |

**Целевые классы (alphabetical, до 8):** `healthy`, `leaf_spot`, `mealybug`, `not_a_plant`, `nutrient_deficiency`, `powdery_mildew`, `scale_insect`, `spider_mites`. Финальное число определяется фильтром `MIN_TRAIN_PER_CLASS = 60` (см. ячейку подготовки данных).

## Что изменилось vs v3.1

**Архитектурно:**
- Detector теперь **одноклассовый (`leaf`)**. В v3.1 у детектора было два класса (`leaf`/`diseased_leaf`), но из-за смещённого PlantDoc в обучении большинство bbox были `diseased_leaf` → детектор почти на любой лист ставил «болен», что закрепляло «всё больное» ещё до классификатора. Решение «болеет ли лист» полностью передано классификатору.
- Classifier — **EfficientNet-B0 @ 256×256** вместо EfficientNetV2-S @ 300×300. На задаче 7-8 классов и ~10-20k фото EffNet-B0 теряет <1pp top-1, но ONNX/CPU инференс в **2-3 раза быстрее** (~25-40 ms vs 90-140 ms на 4-ядерном x86).

**Данные:**
- **PlantVillage (CC-BY-3.0)** — точечная выгрузка через sparse-checkout: только классы, релевантные комнатным (`Tomato___Spider_mites_Two-spotted_spider_mite` — буквально та же *Tetranychus urticae*, что атакует фикус/монстеру; `Squash___Powdery_mildew`, `Cherry___Powdery_mildew`; `Tomato___Septoria_leaf_spot`, `Tomato___Bacterial_spot`).
- **iNaturalist Open Data API** — реальные индоор-фото вредителей по таксонам *Pseudococcidae* (мучнистый червец), *Coccoidea* (щитовка), *Tetranychus urticae* in the wild, *Erysiphales* (мучнистая роса). Фильтр по лицензии `cc0,cc-by,cc-by-sa` + research-grade. Это закрывает главный пробел v3.1: PlantVillage даёт идеальные студийные кадры, а iNat даёт настоящий шум комнатной съёмки.
- **Houseplant Species (CC0)** — фото комнатных. Раньше шли в `healthy` целиком (целое растение в кадре), но Stage-2 учится на кропах листа → distribution mismatch. **Теперь автокроп через свежеобученный детектор**: каждое фото пропускается через YOLOv8n, и в `healthy` идёт центральный лист, а не «всё растение в горшке».
- **PlantDoc (CC-BY 4.0)** — bbox-annotated реальные фото. Используется только для тренировки детектора.
- **COCO val2017 (CC-BY 4.0)** для `not_a_plant` с фильтром plant-like категорий (potted plant, vase, фрукты-овощи).

**Тренировка классификатора:**
- **Только WeightedRandomSampler.** В v3.1 одновременно работали и сэмплер, и inverse-frequency веса в FocalLoss. Это двойная балансировка, которая утрирует редкие классы — главная причина «всё больное».
- **CrossEntropy + label_smoothing=0.1.** FocalLoss убран — он усиливал ту же ошибку. На сбалансированном сэмплером батче простой CE стабильнее.
- **CutMix/MixUp только внутри plant-классов.** `not_a_plant` исключён из перемешивания — иначе пиксели стула попадали в «healthy» с долей `lam`, и класс `not_a_plant` размывался.
- **Detector-aware фильтр здоровых.** Houseplant Species прогоняется через детектор, и в `healthy` копируется только то, где найден реальный лист.

**Постпроцессинг сервера (см. `server/pipeline.py`):**
- `HEALTHY_MIN_CONFIDENCE` ослаблен **0.85 → 0.55**, `HEALTHY_MARGIN` **0.20 → 0.10**.
- Удалён авто-rerouting на runner-up disease — слишком агрессивно ломал «здоровое». Теперь только warning `uncertain_healthy` без подмены класса.
- `primary_box` для классификатора выбирается не по детектору («больной/здоровый»), а пост-фактум: классификатор пробегает по всем bbox и выбирается тот, у которого max(softmax над болезнями) выше.

## Бюджет

T4 free Colab, ~3-3.5 ч end-to-end:
- Скачивание данных + iNat: ~25-35 мин
- YOLOv8n (single class): ~30-40 мин
- Auto-crop healthy: ~3-5 мин
- EfficientNet-B0 (Phase A 8 ep + Phase B 14 ep): ~75-90 мин
- ONNX export + dynamic INT8 quant + sanity-check: ~5 мин
- E2E demo + outputs: ~3 мин

## Как использовать

1. Открыть в Google Colab → `Runtime → Change runtime type → T4 GPU`.
2. (Один раз) загрузить `kaggle.json` для Kaggle API.
3. Run All сверху вниз.
4. В конце скачать `detector.pt`, `classifier.onnx`, `classifier_int8.onnx`, `classes.json` → положить в `server/models/`.


## 1. Установка зависимостей и проверка GPU

In [ ]:
import sys, subprocess

# Colab уже содержит: torch, torchvision, Pillow, opencv-python,
# scikit-learn, matplotlib, tqdm, pyyaml, requests. Их повторная
# установка ничего не обновляет, но крадёт ~60 секунд.
# Нам нужны только: ultralytics (YOLO), kaggle (CLI), и весь ONNX-стек.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'ultralytics>=8.2.0', 'kaggle',
    'onnx>=1.16', 'onnxruntime>=1.17', 'onnxsim>=0.4',
])

# aria2 — для многопоточного скачивания COCO val2017 (1 GB).
# Если apt молчит, упадём на wget — cell 4 проверит наличие.
subprocess.run(
    ['apt-get', 'install', '-y', '-qq', 'aria2'],
    check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

import torch, torchvision, ultralytics, onnxruntime as ort
print(f'PyTorch:       {torch.__version__}')
print(f'TorchVision:   {torchvision.__version__}')
print(f'Ultralytics:   {ultralytics.__version__}')
print(f'ONNXRuntime:   {ort.__version__}')
print(f'CUDA:          {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:           {torch.cuda.get_device_name(0)}')
    print(f'Memory:        {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice: {device}')


## 2. Загружаем датасеты

Все источники свободны для исследовательского/коммерческого использования (CC-BY/CC-BY-SA/CC0/Apache):

| Источник | Лицензия | Что берём | Зачем |
|----------|----------|-----------|-------|
| PlantDoc OD | CC-BY 4.0 | bbox-аннотации листьев в живых кадрах | детектор |
| PlantVillage (sparse) | CC-BY-3.0 | spider_mites, powdery_mildew, leaf_spot | классификатор (clean) |
| iNaturalist Open Data | CC0/CC-BY/CC-BY-SA | реальные фото клеща/червеца/щитовки/росы | классификатор (wild) |
| Houseplant Species (Kaggle) | CC0 | комнатные растения | healthy после авто-кропа |
| COCO val2017 + ann | CC-BY 4.0 | случайные «не-растения» | not_a_plant |

iNaturalist API ограничен 60 req/min. Скрипт уважает rate-limit и кеширует загрузки в `raw/inaturalist/<taxon>/`.

In [ ]:
import os, shutil, subprocess, time, json, re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests

WORKDIR = Path('/content/plantscope')
RAW = WORKDIR / 'raw'
RAW.mkdir(parents=True, exist_ok=True)


def _has(cmd: str) -> bool:
    return subprocess.run(['which', cmd], stdout=subprocess.DEVNULL,
                          stderr=subprocess.DEVNULL).returncode == 0


def _link_or_copy(src: Path, dst: Path) -> None:
    """Hardlink (мгновенно, без расхода места) с откатом на copy2,
    если файлы оказались на разных файловых системах."""
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except (OSError, FileNotFoundError):
        shutil.copy2(src, dst)


def _download_zip(url: str, out: Path) -> None:
    """Многопоточное скачивание через aria2c (если есть), иначе wget."""
    if _has('aria2c'):
        subprocess.check_call([
            'aria2c', '-x', '16', '-s', '16', '-q',
            '--allow-overwrite=true', '--auto-file-renaming=false',
            '-d', str(out.parent), '-o', out.name, url,
        ])
    else:
        subprocess.check_call(['wget', '-q', '-O', str(out), url])


def _unzip(src: Path, dst: Path, only: str | None = None) -> None:
    """Системный unzip быстрее, чем zipfile на больших архивах."""
    cmd = ['unzip', '-q', '-o', str(src), '-d', str(dst)]
    if only:
        cmd.insert(2, only)
    subprocess.check_call(cmd)


# ── 2.1 PlantDoc Object Detection ────────────────────────────────
plantdoc_dir = RAW / 'plantdoc_od'

def fetch_plantdoc():
    if plantdoc_dir.exists():
        return f'PlantDoc уже на диске ({sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg).'
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '-q',
        'https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git',
        str(plantdoc_dir),
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return f'PlantDoc OD: {sum(1 for _ in plantdoc_dir.rglob("*.jpg"))} jpg.'


# ── 2.2 PlantVillage sparse-checkout ─────────────────────────────
PV_FOLDERS = {
    'spider_mites': [
        'raw/color/Tomato___Spider_mites Two-spotted_spider_mite',
    ],
    'powdery_mildew': [
        'raw/color/Squash___Powdery_mildew',
        'raw/color/Cherry_(including_sour)___Powdery_mildew',
    ],
    'leaf_spot': [
        'raw/color/Tomato___Septoria_leaf_spot',
        'raw/color/Tomato___Bacterial_spot',
        'raw/color/Pepper,_bell___Bacterial_spot',
    ],
}
pv_root = RAW / 'plantvillage_repo'
pv_extracted = RAW / 'plantvillage_extracted'

def fetch_plantvillage():
    if pv_extracted.exists() and any(pv_extracted.iterdir()):
        return 'PlantVillage уже распакован.'
    pv_extracted.mkdir(exist_ok=True)
    if not pv_root.exists():
        subprocess.check_call([
            'git', 'clone', '--filter=blob:none', '--no-checkout', '--depth', '1', '-q',
            'https://github.com/spMohanty/PlantVillage-Dataset.git', str(pv_root),
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'sparse-checkout', 'init', '--cone'],
        stdout=subprocess.DEVNULL,
    )
    all_paths = [p for paths in PV_FOLDERS.values() for p in paths]
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'sparse-checkout', 'set'] + all_paths,
        stdout=subprocess.DEVNULL,
    )
    subprocess.check_call(
        ['git', '-C', str(pv_root), 'checkout', '-q'],
        stdout=subprocess.DEVNULL,
    )
    total = 0
    for cls, paths in PV_FOLDERS.items():
        out = pv_extracted / cls
        out.mkdir(exist_ok=True)
        for p in paths:
            src = pv_root / p
            if not src.exists():
                continue
            for f in src.iterdir():
                if f.suffix.lower() in ('.jpg', '.jpeg', '.png'):
                    _link_or_copy(f, out / f.name)  # hardlink — мгновенно
                    total += 1
    return f'PlantVillage: {total} файлов слинковано.'


# ── 2.3 COCO val2017 (~1 GB, через aria2c) ───────────────────────
coco_dir = RAW / 'coco_val2017'

def fetch_coco_images():
    if coco_dir.exists() and any(coco_dir.glob('*.jpg')):
        return f'COCO val2017 уже на диске ({sum(1 for _ in coco_dir.glob("*.jpg"))} jpg).'
    coco_zip = RAW / 'coco_val2017.zip'
    _download_zip('http://images.cocodataset.org/zips/val2017.zip', coco_zip)
    _unzip(coco_zip, RAW)
    extracted = RAW / 'val2017'
    if extracted.exists():
        extracted.rename(coco_dir)
    coco_zip.unlink(missing_ok=True)
    return f'COCO val2017: {sum(1 for _ in coco_dir.glob("*.jpg"))} jpg распаковано.'


coco_ann = RAW / 'annotations' / 'instances_val2017.json'

def fetch_coco_annotations():
    if coco_ann.exists():
        return 'COCO annotations уже на диске.'
    ann_zip = RAW / 'coco_ann.zip'
    _download_zip(
        'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
        ann_zip,
    )
    _unzip(ann_zip, RAW, only='annotations/instances_val2017.json')
    ann_zip.unlink(missing_ok=True)
    return 'COCO annotations распакованы.'


# ── 2.5 iNaturalist Open Data — реальные индоор-фото ─────────────
#
# Раньше: ~3000 фото скачивались последовательно по ~1 секунде каждое
# → ~30+ минут. Теперь: API-листинг страниц остаётся последовательным
# (rate-limit ≤60 req/min на api.inaturalist.org), но _скачивание_
# изображений с CDN (static.inaturalist.org) распараллелено в пуле
# на 16 потоков. Падение времени iNat-фазы: ~30 мин → ~2-4 мин.

INAT_TAXA = {
    'mealybug':         (47651,  'Pseudococcidae',           'mealybug'),
    'scale_insect':     (50189,  'Coccoidea',                'scale_insect'),
    'spider_mite_wild': (124417, 'Tetranychus urticae',      'spider_mites'),
    'powdery_mildew':   (152032, 'Erysiphales',              'powdery_mildew'),
}
INAT_QUERY = {
    'leaf_chlorosis': ('leaf chlorosis houseplant', 'nutrient_deficiency'),
    'leaf_yellowing': ('houseplant yellow leaf',   'nutrient_deficiency'),
}
TARGET_PER_TAXON = 600
INAT_BASE = 'https://api.inaturalist.org/v1/observations'
_HTTP = requests.Session()  # один TCP-keep-alive пул на все запросы


def _inat_download_one(url: str, dst: Path) -> bool:
    if dst.exists():
        return True
    try:
        r = _HTTP.get(url, timeout=15)
    except Exception:
        return False
    if r.status_code != 200 or len(r.content) <= 4096:
        return False
    try:
        dst.write_bytes(r.content)
        return True
    except Exception:
        return False


def _inat_fetch(params: dict, target: int, out_dir: Path) -> int:
    out_dir.mkdir(parents=True, exist_ok=True)
    have = sum(1 for f in out_dir.iterdir()
               if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    if have >= target:
        return have
    page = 1
    saved = have
    # 16 параллельных HTTP-загрузок к CDN — у инатовского кеша свободно.
    with ThreadPoolExecutor(max_workers=16) as pool:
        while saved < target and page <= 30:
            q = dict(params, per_page=200, page=page)
            try:
                r = _HTTP.get(INAT_BASE, params=q, timeout=30)
                if r.status_code != 200:
                    print(f'  iNat HTTP {r.status_code} on page {page}, stopping')
                    break
                data = r.json()
            except Exception as e:
                print(f'  iNat error page {page}: {e}'); break
            results = data.get('results', [])
            if not results:
                break
            # Соберём задания для скачивания всех фото со страницы разом.
            jobs = []
            for obs in results:
                for ph in obs.get('photos', []):
                    lic = (ph.get('license_code') or '').lower()
                    if lic not in ('cc0', 'cc-by', 'cc-by-sa'):
                        continue
                    url = (ph.get('url') or '') \
                        .replace('/square.', '/medium.') \
                        .replace('/small.', '/medium.')
                    if not url:
                        continue
                    ext = '.jpg'
                    m = re.search(r'\.(jpe?g|png)(?:\?|$)', url, re.IGNORECASE)
                    if m:
                        ext = '.' + m.group(1).lower()
                    fn = out_dir / f"{obs['id']}_{ph['id']}{ext}"
                    jobs.append((url, fn))
                    if len(jobs) + saved >= target * 2:  # запас под отказы
                        break
                if len(jobs) + saved >= target * 2:
                    break
            futures = [pool.submit(_inat_download_one, url, fn) for url, fn in jobs]
            for fut in as_completed(futures):
                if fut.result():
                    saved += 1
                if saved >= target:
                    break
            page += 1
            time.sleep(1.1)  # ≤60 req/min на API-листинг
    return saved


inat_root = RAW / 'inaturalist'

def fetch_inaturalist():
    inat_root.mkdir(exist_ok=True)
    msgs: list[str] = []
    # Все шесть запросов выполняем последовательно (чтобы не ловить
    # rate-limit на api.inaturalist.org), но КАЖДАЯ страница — это
    # 16-поточная закачка картинок с CDN.
    for key, (tid, label, cls) in INAT_TAXA.items():
        n = _inat_fetch({
            'taxon_id': tid,
            'photo_license': 'cc0,cc-by,cc-by-sa',
            'quality_grade': 'research',
            'identified': 'true',
        }, TARGET_PER_TAXON, inat_root / key)
        msgs.append(f'iNat {label} → {cls}: {n}')
    for key, (q, cls) in INAT_QUERY.items():
        n = _inat_fetch({
            'q': q,
            'photo_license': 'cc0,cc-by,cc-by-sa',
            'quality_grade': 'research',
        }, TARGET_PER_TAXON // 2, inat_root / key)
        msgs.append(f'iNat q="{q}" → {cls}: {n}')
    return '\n  '.join(msgs)


# ── Параллельный fetch — все источники одновременно ──────────────
JOBS = [
    ('plantdoc',     fetch_plantdoc),
    ('plantvillage', fetch_plantvillage),
    ('coco_images',  fetch_coco_images),
    ('coco_ann',     fetch_coco_annotations),
    ('inaturalist',  fetch_inaturalist),
]
print(f'Параллельно качаем {len(JOBS)} источника(ов)…  '
      f'(aria2c={"yes" if _has("aria2c") else "no, wget fallback"})')
t0 = time.time()
with ThreadPoolExecutor(max_workers=len(JOBS)) as pool:
    futures = {pool.submit(fn): name for name, fn in JOBS}
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            print(f'  ✓ {name}: {fut.result()}')
        except Exception as e:
            print(f'  ✗ {name}: {e}')
print(f'\nDownload total: {time.time() - t0:.1f}s')


# ── 2.4 Houseplant Species (Kaggle) — после параллельной фазы.
# Kaggle CLI требует интерактивную загрузку kaggle.json через ipywidgets,
# которая не дружит с фоновыми потоками; делаем последовательно в конце.
hp_dir = RAW / 'houseplant'
if not hp_dir.exists():
    kj = Path.home() / '.kaggle' / 'kaggle.json'
    if not kj.exists():
        try:
            from google.colab import files  # type: ignore
            print('Загрузите kaggle.json (Kaggle → Account → Create API Token)…')
            up = files.upload()
            kj.parent.mkdir(parents=True, exist_ok=True)
            for n, d in up.items():
                kj.write_bytes(d)
            kj.chmod(0o600)
        except Exception as e:
            print(f'kaggle.json не получен: {e}')
    if kj.exists():
        try:
            subprocess.check_call(['kaggle', 'datasets', 'download', '-d',
                'kacpergregorowicz/house-plant-species', '-p', str(hp_dir),
                '--unzip', '-q'])
        except Exception as e:
            print(f'Kaggle Houseplant Species: {e}')
n_hp = sum(1 for _ in hp_dir.rglob('*.jpg')) + sum(1 for _ in hp_dir.rglob('*.jpeg')) \
       if hp_dir.exists() else 0
print(f'Houseplant Species: {n_hp} images')

print('\nДанные готовы. Источники:')
print(f'  PlantDoc OD     : {plantdoc_dir}')
print(f'  PlantVillage    : {pv_extracted}')
print(f'  COCO val2017    : {coco_dir}')
print(f'  Houseplant      : {hp_dir}')
print(f'  iNaturalist     : {inat_root}')


## 3. YOLO-датасет: один класс `leaf`

В v4 детектор **одноклассовый**: он только локализует листья. Решение «болеет ли» делает классификатор. Такая декомпозиция убирает главный bias старой версии (детектор почти всегда ставил `diseased_leaf`).

PlantDoc даёт PASCAL VOC XML на 27 классов; все они мапятся в `leaf`. 85/15 train/val.

In [ ]:
import os
import xml.etree.ElementTree as ET
import shutil, random
from pathlib import Path


def _link_or_copy(src, dst) -> None:
    """Hardlink (мгновенно) с откатом на copy2 при ошибке."""
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except (OSError, FileNotFoundError):
        shutil.copy2(src, dst)

YOLO_DIR = WORKDIR / 'detector'
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for sub in ('images/train', 'images/val', 'labels/train', 'labels/val'):
    (YOLO_DIR / sub).mkdir(parents=True, exist_ok=True)

random.seed(42)
pairs: list[tuple[Path, Path]] = []
for split_name in ('TRAIN', 'TEST'):
    split_dir = plantdoc_dir / split_name
    if not split_dir.exists():
        continue
    for xml in split_dir.rglob('*.xml'):
        for ext in ('.jpg', '.JPG', '.jpeg', '.png'):
            img = xml.with_suffix(ext)
            if img.exists():
                pairs.append((img, xml))
                break
print(f'PlantDoc XML pairs: {len(pairs)}')
random.shuffle(pairs)
cut = int(len(pairs) * 0.85)
train_pairs, val_pairs = pairs[:cut], pairs[cut:]

def write_yolo(pair_list, split):
    kept = 0
    for img, xml in pair_list:
        try:
            root = ET.parse(xml).getroot()
        except ET.ParseError:
            continue
        size = root.find('size')
        if size is None:
            continue
        W = float(size.findtext('width') or 0)
        H = float(size.findtext('height') or 0)
        if W <= 0 or H <= 0:
            continue
        lines = []
        for obj in root.findall('object'):
            name = (obj.findtext('name') or '').lower()
            if 'leaf' not in name:
                continue
            bb = obj.find('bndbox')
            if bb is None:
                continue
            xmin = float(bb.findtext('xmin') or 0)
            ymin = float(bb.findtext('ymin') or 0)
            xmax = float(bb.findtext('xmax') or 0)
            ymax = float(bb.findtext('ymax') or 0)
            if xmax <= xmin or ymax <= ymin:
                continue
            cx = ((xmin + xmax) / 2) / W
            cy = ((ymin + ymax) / 2) / H
            bw = (xmax - xmin) / W
            bh = (ymax - ymin) / H
            cx, cy = min(max(cx, 0), 1), min(max(cy, 0), 1)
            bw, bh = min(max(bw, 0.001), 1), min(max(bh, 0.001), 1)
            # одноклассовый детектор — class_id всегда 0
            lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
        if not lines:
            continue
        _link_or_copy(img, YOLO_DIR / f'images/{split}' / img.name)
        (YOLO_DIR / f'labels/{split}' / (img.stem + '.txt')).write_text(
            '\n'.join(lines), encoding='utf-8')
        kept += 1
    return kept

n_tr = write_yolo(train_pairs, 'train')
n_vl = write_yolo(val_pairs, 'val')
print(f'YOLO train={n_tr}  val={n_vl}')

data_yaml = YOLO_DIR / 'data.yaml'
data_yaml.write_text(
    f'path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnc: 1\nnames: [leaf]\n',
    encoding='utf-8')
print('data.yaml →', data_yaml)

## 4. Тренируем YOLOv8n (Stage 1)

Одноклассовый детектор сходится быстрее многоклассового. T4: ~30-40 мин на 60 эпох. Ключевые решения:

- **`cache='ram'`** — PlantDoc (~2.5k jpeg) влезает в RAM Colab, экономит 30-40% wall-clock.
- **`mosaic=1.0` + `mixup=0.1`** — устойчивость к шумным комнатным фото.
- **`close_mosaic=10`** — финальные эпохи без mosaic, стабильные bbox-границы.
- **`hsv_*`** — устойчивость к индоор-освещению (тёплая лампа, окно).
- **AMP + cos_lr + patience=12** — стандартный efficient-recipe.

In [ ]:
from ultralytics import YOLO
import shutil

det_model = YOLO('yolov8n.pt')

det_results = det_model.train(
    data=str(data_yaml),
    epochs=60,
    imgsz=640,
    batch=32,
    patience=12,
    cos_lr=True,
    amp=True,
    cache='ram',
    workers=4,
    save_period=-1,
    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    project=str(WORKDIR / 'yolo_runs'),
    name='detector',
    exist_ok=True,
    verbose=True,
)

best_pt = WORKDIR / 'yolo_runs' / 'detector' / 'weights' / 'best.pt'
assert best_pt.exists(), f'YOLO best.pt не сохранён в {best_pt}'
MODELS_OUT = WORKDIR / 'out'
MODELS_OUT.mkdir(exist_ok=True)
shutil.copy2(best_pt, MODELS_OUT / 'detector.pt')

# Ультралитика напрямую экспортит ONNX. NMS встраивается в граф.
det_model.export(format='onnx', imgsz=640, opset=13, simplify=True, dynamic=False)
onnx_src = best_pt.with_suffix('.onnx')
if onnx_src.exists():
    shutil.copy2(onnx_src, MODELS_OUT / 'detector.onnx')
    print('Detector ONNX →', MODELS_OUT / 'detector.onnx')

metrics = det_model.val(data=str(data_yaml), imgsz=640, plots=False, verbose=False)
print(f'mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}')

## 5. Классификаторный датасет (Stage 2)

Целевые классы: `blight` 🚫, `leaf_mold` 🚫, `mosaic_virus` 🚫, `rust` 🚫 — убраны (полевые культуры). Остались **8 houseplant-релевантных**:

| Класс | Откуда |
|-------|--------|
| `healthy`              | Houseplant Species (CC0) — авто-кроп через детектор |
| `not_a_plant`          | COCO val2017 минус plant-like категории |
| `spider_mites`         | PlantVillage Tomato___Spider_mites + iNat *Tetranychus urticae* |
| `mealybug`             | iNat *Pseudococcidae* |
| `scale_insect`         | iNat *Coccoidea* |
| `powdery_mildew`       | PlantVillage Squash/Cherry + iNat *Erysiphales* |
| `leaf_spot`            | PlantVillage Septoria/Bacterial spot |
| `nutrient_deficiency`  | iNat q="leaf chlorosis", "houseplant yellow leaf" |

**Авто-кроп healthy через детектор** — главный фикс distribution mismatch v3.1: раньше Houseplant Species подавался целыми растениями, а классификатор обучался на кропах листа из PlantDoc/PlantVillage.

In [ ]:
import os, shutil, random, json
from pathlib import Path
from PIL import Image
import numpy as np
from ultralytics import YOLO

CLS_DIR = WORKDIR / 'classifier'
if CLS_DIR.exists():
    shutil.rmtree(CLS_DIR)

TARGET_CLASSES = [
    'healthy', 'leaf_spot', 'mealybug', 'not_a_plant',
    'nutrient_deficiency', 'powdery_mildew', 'scale_insect', 'spider_mites',
]
random.seed(42)
VAL_RATIO = 0.15
stats = {c: {'train': 0, 'val': 0} for c in TARGET_CLASSES}
for split in ('train', 'val'):
    for cls in TARGET_CLASSES:
        (CLS_DIR / split / cls).mkdir(parents=True, exist_ok=True)

def _link_or_copy(src: Path, dst: Path) -> None:
    """Hardlink (мгновенно) с откатом на copy2 при ошибке."""
    try:
        os.link(src, dst)
    except (OSError, FileNotFoundError):
        shutil.copy2(src, dst)


def put(img_path: Path, target_cls: str):
    if target_cls not in stats:
        return
    split = 'val' if random.random() < VAL_RATIO else 'train'
    ext = img_path.suffix.lower() or '.jpg'
    dst = CLS_DIR / split / target_cls / f'{target_cls}_{stats[target_cls][split]:06d}{ext}'
    try:
        _link_or_copy(img_path, dst)
        stats[target_cls][split] += 1
    except Exception:
        pass

def put_pil(img: Image.Image, target_cls: str):
    if target_cls not in stats:
        return
    split = 'val' if random.random() < VAL_RATIO else 'train'
    dst = CLS_DIR / split / target_cls / f'{target_cls}_{stats[target_cls][split]:06d}.jpg'
    try:
        img.convert('RGB').save(dst, 'JPEG', quality=90)
        stats[target_cls][split] += 1
    except Exception:
        pass

# ── 5.1 PlantVillage кропы ──
PV_TO_CLASS = {
    'spider_mites': 'spider_mites',
    'powdery_mildew': 'powdery_mildew',
    'leaf_spot': 'leaf_spot',
}
PV_CAP = {  # ограничиваем, иначе PlantVillage задавит iNat-данные
    'spider_mites': 800,
    'powdery_mildew': 800,
    'leaf_spot': 1200,
}
for pv_cls, dst_cls in PV_TO_CLASS.items():
    src_dir = pv_extracted / pv_cls
    if not src_dir.exists():
        continue
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    for img in imgs[:PV_CAP[pv_cls]]:
        put(img, dst_cls)
    print(f'PV {pv_cls} → {dst_cls}: {min(len(imgs), PV_CAP[pv_cls])}')

# ── 5.2 iNaturalist (реальные индоор-фото) ──
INAT_MAP = {
    'mealybug':         'mealybug',
    'scale_insect':     'scale_insect',
    'spider_mite_wild': 'spider_mites',
    'powdery_mildew':   'powdery_mildew',
    'leaf_chlorosis':   'nutrient_deficiency',
    'leaf_yellowing':   'nutrient_deficiency',
}
INAT_CAP = 600  # на каждый ключ
for key, dst_cls in INAT_MAP.items():
    src_dir = inat_root / key
    if not src_dir.exists():
        continue
    imgs = [p for p in src_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    random.shuffle(imgs)
    for img in imgs[:INAT_CAP]:
        put(img, dst_cls)
    print(f'iNat {key} → {dst_cls}: {min(len(imgs), INAT_CAP)}')

# ── 5.3 Houseplant Species → healthy (с авто-кропом через детектор) ──
#
# КРИТИЧНЫЙ ФИКС v4: раньше Houseplant Species копировалось как целое
# растение в горшке, а классификатор учится на кропах листа из PD/PV.
# Несоответствие распределений → модель плохо распознавала «здоровое»
# на реальных индоор-фото листа в кадре.
#
# Решение: пропускаем каждое фото через свежеобученный детектор и
# берём центральный bbox с conf>0.4. Если детектор пуст — пропускаем.
HP_TARGET_HEALTHY = 1500

if hp_dir.exists():
    det = YOLO(str(MODELS_OUT / 'detector.pt'))
    hp_imgs = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG'):
        hp_imgs.extend(hp_dir.rglob(ext))
    random.shuffle(hp_imgs)
    saved_full = 0
    saved_crop = 0
    for img_path in hp_imgs:
        if stats['healthy']['train'] + stats['healthy']['val'] >= HP_TARGET_HEALTHY:
            break
        try:
            pil = Image.open(img_path).convert('RGB')
        except Exception:
            continue
        # детектор хочет numpy
        arr = np.array(pil)
        try:
            res = det.predict(arr, imgsz=640, conf=0.4, verbose=False, device=device)[0]
        except Exception:
            continue
        if res.boxes is None or len(res.boxes) == 0:
            # Нет видимого листа — НЕ копируем; такие фото только размывают healthy.
            continue
        # центральный bbox (по площади и близости к центру)
        boxes = res.boxes.xyxy.cpu().numpy()
        H, W = arr.shape[:2]
        cx, cy = W / 2, H / 2
        def score(b):
            x1, y1, x2, y2 = b
            area = (x2 - x1) * (y2 - y1)
            d2 = ((x1 + x2) / 2 - cx) ** 2 + ((y1 + y2) / 2 - cy) ** 2
            return area - 0.3 * d2
        b = max(boxes, key=score)
        x1, y1, x2, y2 = (int(round(v)) for v in b)
        # padding
        pad_x = int((x2 - x1) * 0.1)
        pad_y = int((y2 - y1) * 0.1)
        x1, y1 = max(0, x1 - pad_x), max(0, y1 - pad_y)
        x2, y2 = min(W, x2 + pad_x), min(H, y2 + pad_y)
        if x2 - x1 < 32 or y2 - y1 < 32:
            continue
        crop = pil.crop((x1, y1, x2, y2))
        put_pil(crop, 'healthy')
        saved_crop += 1
    print(f'Houseplant Species → healthy: {saved_crop} cropped via detector')

# ── 5.4 COCO → not_a_plant с фильтром plant-like категорий ──
PLANT_LIKE_COCO_IDS = {
    64,  # potted plant
    86,  # vase
    56,  # broccoli
    50, 51, 52, 53, 54, 55, 57, 58, 59, 60, 61,  # food (apple/banana/orange…)
}
banned: set[int] = set()
if coco_ann.exists():
    coco_meta = json.loads(coco_ann.read_text(encoding='utf-8'))
    for a in coco_meta.get('annotations', []):
        if a.get('category_id') in PLANT_LIKE_COCO_IDS:
            banned.add(a['image_id'])
    print(f'COCO ban-list: {len(banned)} images excluded')

def cid(p: Path) -> int:
    try:
        return int(p.stem)
    except ValueError:
        return -1

coco_imgs = [p for p in coco_dir.glob('*.jpg') if cid(p) not in banned]
random.shuffle(coco_imgs)
COCO_CAP = 1200
for img in coco_imgs[:COCO_CAP]:
    put(img, 'not_a_plant')
print(f'COCO → not_a_plant: {min(len(coco_imgs), COCO_CAP)}')

# ── 5.5 Защитный фильтр: классы с <60 train отсекаем ──
MIN_TRAIN_PER_CLASS = 60
dropped: list[str] = []
for cls in list(TARGET_CLASSES):
    if stats[cls]['train'] < MIN_TRAIN_PER_CLASS:
        dropped.append(cls)
        for split in ('train', 'val'):
            d = CLS_DIR / split / cls
            if d.exists():
                shutil.rmtree(d)
        TARGET_CLASSES.remove(cls)
        stats.pop(cls, None)

print('\nРаспределение по классам:')
for cls in TARGET_CLASSES:
    print(f'  {cls:<22}  train={stats[cls]["train"]:>4}  val={stats[cls]["val"]:>4}')
print(f'\nИтого: {sum(s["train"] for s in stats.values())} train / '
      f'{sum(s["val"] for s in stats.values())} val  ({len(TARGET_CLASSES)} классов)')
if dropped:
    print('\n⚠ Отсечены (<60 train):', dropped)
    print('  classes.json и чек-пойнт выйдут с уменьшённым числом классов;')
    print('  сервер примет это автоматически (classifier.py обрезает CLASS_NAMES).')

## 6. EfficientNet-B0 — обучение классификатора

**Ключевые отличия v4 от v3.1:**

| Аспект | v3.1 | v4 |
|--------|------|-----|
| Backbone        | EfficientNetV2-S (21M, 300px) | **EfficientNet-B0 (5.3M, 256px)** |
| Loss            | FocalLoss + inverse-freq weights | **CrossEntropy + label_smoothing 0.1** |
| Балансировка    | WeightedRandomSampler **+** class_weights | **Только WeightedRandomSampler** |
| CutMix/MixUp    | На любом батче, смешивает с not_a_plant | **Только внутри plant-классов** |
| TTA             | h-flip | h-flip |
| EMA decay       | 0.999 | 0.999 |

Двойная балансировка (sampler+weights) утрировала редкие классы и душила healthy — главная причина «всё больное» в v3.1. Здесь — одна балансировка, и она у сэмплера.

Phase A (head only, 8 ep) → Phase B (fine-tune features.5/6/7+head, 14 ep). T4: ~75-90 мин.

In [ ]:
import copy, io, json, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE = 256
BATCH_SIZE = 64
NUM_WORKERS = 4
torch.backends.cudnn.benchmark = True


class RandomJPEGCompression:
    def __init__(self, p=0.5, q_range=(40, 95)):
        self.p, self.q_min, self.q_max = p, q_range[0], q_range[1]
    def __call__(self, img):
        if random.random() > self.p:
            return img
        q = random.randint(self.q_min, self.q_max)
        buf = io.BytesIO()
        img.convert('RGB').save(buf, format='JPEG', quality=q)
        buf.seek(0)
        return Image.open(buf).convert('RGB')


train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(20),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.ColorJitter(0.3, 0.3, 0.25, 0.05),
    transforms.RandomAffine(degrees=0, translate=(0.06, 0.06), shear=6),
    transforms.GaussianBlur(3, sigma=(0.1, 1.5)),
    transforms.TrivialAugmentWide(),
    RandomJPEGCompression(p=0.4, q_range=(45, 95)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.18)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(str(CLS_DIR / 'train'), transform=train_tf)
val_ds   = datasets.ImageFolder(str(CLS_DIR / 'val'),   transform=val_tf)
class_names = list(train_ds.classes)
num_classes = len(class_names)
not_a_plant_idx = class_names.index('not_a_plant') if 'not_a_plant' in class_names else -1
print('Classes:', class_names, '| not_a_plant idx =', not_a_plant_idx)

# WeightedRandomSampler — единственный механизм балансировки (без class_weights в loss).
counts = [0] * num_classes
for _, y in train_ds.samples:
    counts[y] += 1
total = sum(counts)
weights = [total / (c or 1) for c in counts]
sample_w = [weights[y] for _, y in train_ds.samples]
sampler = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=4, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, prefetch_factor=2)

net = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, num_classes)
for name, p in net.named_parameters():
    p.requires_grad = name.startswith('classifier.')
net = net.to(memory_format=torch.channels_last).to(device)


# ── CutMix/MixUp с защитой от not_a_plant ──
# Не смешиваем с фоновым классом — иначе пиксели стула попадают
# в healthy/leaf_spot с весом lam, и модель путается, что такое not_a_plant.
def _rand_bbox(w, h, lam):
    cut_rat = math.sqrt(1.0 - lam)
    cw = int(w * cut_rat); ch = int(h * cut_rat)
    cx = random.randint(0, w - 1); cy = random.randint(0, h - 1)
    return (max(0, cx - cw // 2), max(0, cy - ch // 2),
            min(w, cx + cw // 2), min(h, cy + ch // 2))

def _safe_perm(y):
    # Перестановка не смешивает not_a_plant с plant-классами и наоборот.
    if not_a_plant_idx < 0:
        return torch.randperm(y.size(0), device=y.device)
    is_nap = (y == not_a_plant_idx)
    perm = torch.arange(y.size(0), device=y.device)
    plant_idx = torch.where(~is_nap)[0]
    nap_idx = torch.where(is_nap)[0]
    if len(plant_idx) > 1:
        perm[plant_idx] = plant_idx[torch.randperm(len(plant_idx), device=y.device)]
    if len(nap_idx) > 1:
        perm[nap_idx] = nap_idx[torch.randperm(len(nap_idx), device=y.device)]
    return perm

def cutmix_batch(x, y, alpha=1.0):
    lam = float(np.random.beta(alpha, alpha))
    perm = _safe_perm(y)
    x1, y1, x2, y2 = _rand_bbox(x.size(3), x.size(2), lam)
    x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam = 1.0 - ((x2 - x1) * (y2 - y1)) / (x.size(3) * x.size(2))
    return x, y, y[perm], lam

def mixup_batch(x, y, alpha=0.2):
    lam = float(np.random.beta(alpha, alpha))
    perm = _safe_perm(y)
    return lam * x + (1.0 - lam) * x[perm], y, y[perm], lam


# Простой CE+LS, без class_weights — балансировка делается сэмплером.
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

use_amp = device.type == 'cuda'
try:
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    autocast_ctx = lambda: torch.amp.autocast('cuda', enabled=use_amp)  # noqa: E731
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    autocast_ctx = lambda: torch.cuda.amp.autocast(enabled=use_amp)  # noqa: E731


class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.ema = copy.deepcopy(model).eval()
        for p in self.ema.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for ep, p in zip(self.ema.parameters(), model.parameters()):
            ep.mul_(d).add_(p.detach(), alpha=1.0 - d)
        for eb, b in zip(self.ema.buffers(), model.buffers()):
            eb.copy_(b)

ema = ModelEMA(net, decay=0.999)
MODELS_OUT = WORKDIR / 'out'; MODELS_OUT.mkdir(exist_ok=True)

CUTMIX_PROB = 0.5

def make_opt_sched(params, lr, epochs):
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    warm = optim.lr_scheduler.LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=1)
    cos = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs - 1, 1))
    return opt, optim.lr_scheduler.SequentialLR(opt, [warm, cos], milestones=[1])

def _tta_predict(model_to_eval, x):
    with autocast_ctx():
        p1 = F.softmax(model_to_eval(x), dim=1)
        p2 = F.softmax(model_to_eval(torch.flip(x, dims=[3])), dim=1)
    return (p1 + p2) * 0.5

def run_train_epoch(loader, optimizer):
    net.train()
    loss_sum = correct = total = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        use_mix = CUTMIX_PROB > 0 and random.random() < CUTMIX_PROB
        if use_mix:
            if random.random() < 0.7:
                x, y_a, y_b, lam = cutmix_batch(x, y, alpha=1.0)
            else:
                x, y_a, y_b, lam = mixup_batch(x, y, alpha=0.2)
        with autocast_ctx():
            logits = net(x)
            if use_mix:
                loss = lam * crit(logits, y_a) + (1.0 - lam) * crit(logits, y_b)
            else:
                loss = crit(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ema.update(net)
        loss_sum += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return loss_sum / total, correct / total

@torch.no_grad()
def run_val_epoch(model_to_eval):
    model_to_eval.eval()
    loss_sum = correct = total = 0
    for x, y in val_loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(device, non_blocking=True)
        probs = _tta_predict(model_to_eval, x)
        loss = F.nll_loss(torch.log(probs.clamp_min(1e-8)), y)
        loss_sum += loss.item() * x.size(0)
        correct += (probs.argmax(1) == y).sum().item()
        total += x.size(0)
    return loss_sum / total, correct / total

best_acc = 0.0
best_state = None
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def phase(lbl, epochs, lr):
    global best_acc, best_state
    opt, sch = make_opt_sched([p for p in net.parameters() if p.requires_grad], lr, epochs)
    print(f'\n=== {lbl} ===')
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_l, tr_a = run_train_epoch(train_loader, opt)
        vl_l, vl_a = run_val_epoch(ema.ema)
        sch.step()
        history['train_loss'].append(tr_l); history['train_acc'].append(tr_a)
        history['val_loss'].append(vl_l);   history['val_acc'].append(vl_a)
        print(f'  {ep}/{epochs}  train_loss={tr_l:.3f} acc={tr_a:.3f}  '
              f'val_loss={vl_l:.3f} acc={vl_a:.3f}  '
              f'lr={opt.param_groups[0]["lr"]:.2e}  ({time.time()-t0:.1f}s)')
        if vl_a > best_acc:
            best_acc = vl_a
            best_state = {k: v.detach().cpu().clone() for k, v in ema.ema.state_dict().items()}
            torch.save(best_state, MODELS_OUT / 'best.pth')

phase('Phase A — head only', epochs=8, lr=1e-3)

for name, p in net.named_parameters():
    p.requires_grad = (
        name.startswith('features.5') or
        name.startswith('features.6') or
        name.startswith('features.7') or
        name.startswith('classifier.')
    )
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'Phase B — fine-tune ({trainable:,} trainable params)')
phase('Phase B — fine-tune', epochs=14, lr=5e-5)

ema.ema.load_state_dict(best_state)
net.load_state_dict(best_state)
print(f'\nBest val accuracy: {best_acc:.4f}  ({best_acc*100:.1f}%)')

torch.save(best_state, MODELS_OUT / 'classifier.pth')

# Per-class отчёт обязателен — иначе модель «всегда healthy» неотличима от настоящей.
all_pred, all_true = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device, non_blocking=True, memory_format=torch.channels_last)
        probs = _tta_predict(ema.ema, x)
        all_pred.extend(probs.argmax(1).cpu().tolist())
        all_true.extend(y.tolist())

print('\n── Per-class metrics (val, TTA) ──')
print(classification_report(all_true, all_pred, target_names=class_names,
                            digits=3, zero_division=0))
cm = confusion_matrix(all_true, all_pred, labels=list(range(num_classes)))
print('Confusion matrix:')
print('            ' + ' '.join(f'{n[:8]:>9}' for n in class_names))
for name, row in zip(class_names, cm, strict=False):
    print(f'{name:>12} ' + ' '.join(f'{v:>9}' for v in row))

## 7. Экспорт ONNX + Dynamic INT8 квантизация

Сервер крутит классификатор на CPU, поэтому экспортируем **EfficientNet-B0 в ONNX FP32** (через `torch.onnx.export`) и сразу же делаем **dynamic INT8 quantization** через `onnxruntime.quantization.quantize_dynamic`. Только веса в INT8, активации остаются FP32 — это безопасно для SiLU-/Swish-сетей (статическая квантизация EfficientNet с SiLU деградирует, dynamic — нет).

Ожидаемые размеры/латентности на 4-ядерном x86 ONNX Runtime CPU EP:

| Файл | Размер | Latency (batch=1, 256×256) |
|------|--------|----------------------------|
| `classifier.pth` (PyTorch eager FP32) | ~22 MB | ~75-95 ms |
| `classifier.onnx` (ONNX FP32)         | ~21 MB | ~30-45 ms |
| `classifier_int8.onnx` (Dynamic INT8) | ~6-8 MB | ~18-30 ms |

Делаем sanity-check: пропускаем val-batch через ONNX FP32 и через INT8, сравниваем top-1 accuracy с PyTorch — отклонение ≤1.5pp считаем приемлемым.

In [ ]:
import shutil, json
import torch
import numpy as np
import onnx
from pathlib import Path

# 1) Экспорт FP32 в ONNX
ONNX_FP32 = MODELS_OUT / 'classifier.onnx'
ONNX_INT8 = MODELS_OUT / 'classifier_int8.onnx'

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
ema.ema.eval()
torch.onnx.export(
    ema.ema.cpu(), dummy.cpu(),
    str(ONNX_FP32),
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch'}, 'logits': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)
ema.ema.to(device)
print('ONNX FP32 →', ONNX_FP32, f'({ONNX_FP32.stat().st_size/1e6:.1f} MB)')

# Sanity: загрузка + проверка структуры
m = onnx.load(str(ONNX_FP32))
onnx.checker.check_model(m)

# 2) Симплификация графа (если получится — иногда падает на dynamic axes)
try:
    from onnxsim import simplify
    m_simp, ok = simplify(m)
    if ok:
        onnx.save(m_simp, str(ONNX_FP32))
        print('ONNX simplified OK')
except Exception as e:
    print(f'onnxsim skipped: {e}')

# 3) Dynamic INT8 квантизация (только веса, активации FP32 — безопасно для SiLU)
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic(
    model_input=str(ONNX_FP32),
    model_output=str(ONNX_INT8),
    weight_type=QuantType.QInt8,
    per_channel=True,  # критично для EfficientNet — снижает дрейф
)
print('ONNX INT8 →', ONNX_INT8, f'({ONNX_INT8.stat().st_size/1e6:.1f} MB)')

# 4) Sanity-check: TTA accuracy на val через ONNX FP32 и INT8
import onnxruntime as ort

def onnx_eval(path):
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    correct = 0
    n = 0
    for x, y in val_loader:
        x_np = x.numpy().astype(np.float32)
        logits = sess.run(None, {inp: x_np})[0]
        pred = logits.argmax(1)
        correct += int((pred == y.numpy()).sum())
        n += len(y)
    return correct / n

acc_fp32 = onnx_eval(ONNX_FP32)
acc_int8 = onnx_eval(ONNX_INT8)
print(f'\nTorch  EMA+TTA val acc : {best_acc:.4f}')
print(f'ONNX FP32 (no TTA)     : {acc_fp32:.4f}')
print(f'ONNX INT8 (no TTA)     : {acc_int8:.4f}')

# 5) classes.json — публикуется вместе с весами
classes_meta = {
    '_comment': 'PlantScope v4 — 8 houseplant-relevant classes. Stage-1 detector is single-class (leaf).',
    'model_version': '4.0.0',
    'architecture': {
        'detector': 'YOLOv8n (single-class leaf)',
        'classifier': 'EfficientNet-B0',
    },
    'num_classes': num_classes,
    'class_names': class_names,
    'detector_classes': ['leaf'],
    'val_accuracy_torch_tta': round(best_acc, 4),
    'val_accuracy_onnx_fp32': round(acc_fp32, 4),
    'val_accuracy_onnx_int8': round(acc_int8, 4),
    'training': {
        'img_size': IMG_SIZE,
        'batch_size': BATCH_SIZE,
        'cutmix_prob': CUTMIX_PROB,
        'loss': 'CrossEntropy(label_smoothing=0.1)',
        'sampler': 'WeightedRandomSampler (single balancing source)',
        'tta': 'hflip',
        'ema_decay': 0.999,
        'cutmix_safe': 'no mixing across not_a_plant ↔ plant classes',
    },
}
(MODELS_OUT / 'classes.json').write_text(
    json.dumps(classes_meta, ensure_ascii=False, indent=2), encoding='utf-8')
print('classes.json →', MODELS_OUT / 'classes.json')

## 8. End-to-end demo

YOLOv8 → bbox листа → классификатор (через ONNX INT8 — так пойдёт в продакшен) → подпись.

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
from ultralytics import YOLO

det = YOLO(str(MODELS_OUT / 'detector.pt'))
sess = ort.InferenceSession(str(MODELS_OUT / 'classifier_int8.onnx'),
                            providers=['CPUExecutionProvider'])
inp_name = sess.get_inputs()[0].name
norm_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
norm_std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(crop: Image.Image) -> np.ndarray:
    arr = np.array(crop.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.float32) / 255.0
    arr = (arr - norm_mean) / norm_std
    return arr.transpose(2, 0, 1)[None].astype(np.float32)

val_images: list[tuple[Path, str]] = []
for cls in class_names:
    for img in (CLS_DIR / 'val' / cls).iterdir():
        val_images.append((img, cls))
random.shuffle(val_images)
demo = val_images[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (path, true_cls) in enumerate(demo):
    ax = axes[i // 3][i % 3]
    img = Image.open(path).convert('RGB')
    arr = np.array(img)
    res = det.predict(arr, imgsz=640, conf=0.25, verbose=False)[0]
    ax.imshow(arr)
    pred_cls, pred_conf = 'not_a_plant', 0.0
    if res.boxes is not None and len(res.boxes):
        boxes = res.boxes.xyxy.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        # Главный bbox — тот, на котором классификатор даёт max уверенность
        # любой не-healthy/не-not_a_plant класса. Если все низкие — берём
        # самый уверенный по детектору.
        best_box = None; best_disease_conf = -1.0; best_pred = None
        for j, (x1, y1, x2, y2) in enumerate(boxes):
            crop = img.crop((int(x1), int(y1), int(x2), int(y2)))
            x_in = preprocess(crop)
            logits = sess.run(None, {inp_name: x_in})[0][0]
            p = np.exp(logits - logits.max()); p /= p.sum()
            disease_score = max(
                p[k] for k, n in enumerate(class_names)
                if n not in ('healthy', 'not_a_plant')
            ) if num_classes > 2 else 0.0
            if disease_score > best_disease_conf:
                best_disease_conf = disease_score
                best_box = (x1, y1, x2, y2)
                best_pred = (class_names[int(p.argmax())], float(p.max()))
        if best_box is not None:
            x1, y1, x2, y2 = best_box
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                       fill=False, edgecolor='lime', linewidth=2.5))
            pred_cls, pred_conf = best_pred
    ok = pred_cls == true_cls
    ax.set_title(f'Pred: {pred_cls} ({pred_conf:.0%})\nTrue: {true_cls}',
                 color='green' if ok else 'red', fontsize=10)
    ax.axis('off')

plt.suptitle('PlantScope v4: YOLOv8n (leaf bbox) + EfficientNet-B0 INT8', fontsize=13)
plt.tight_layout()
plt.show()

## 9. Скачать веса

Положите в `server/models/`:
- `detector.pt`            — YOLOv8n (PyTorch, для совместимости/training)
- `detector.onnx`          — YOLOv8n ONNX (рекомендуется для production)
- `classifier.pth`         — EfficientNet-B0 (PyTorch fallback)
- `classifier.onnx`        — EfficientNet-B0 ONNX FP32
- `classifier_int8.onnx`   — EfficientNet-B0 ONNX dynamic INT8 ⭐ default for server
- `classes.json`           — таксономия + метрики

In [ ]:
import os
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    colab = True
except ImportError:
    colab = False

artifacts = [
    MODELS_OUT / 'detector.pt',
    MODELS_OUT / 'detector.onnx',
    MODELS_OUT / 'classifier.pth',
    MODELS_OUT / 'classifier.onnx',
    MODELS_OUT / 'classifier_int8.onnx',
    MODELS_OUT / 'classes.json',
]
print('Артефакты:')
for p in artifacts:
    if p.exists():
        print(f'  {p.name:<26} {p.stat().st_size / 1e6:>6.2f} MB')
    else:
        print(f'  {p.name:<26}  (отсутствует)')

if colab:
    for p in artifacts:
        if p.exists():
            files.download(str(p))
else:
    print('\nФайлы лежат в', MODELS_OUT)

### (Опционально) Сохранить на Google Drive

In [ ]:
try:
    from google.colab import drive  # type: ignore
    import shutil
    drive.mount('/content/drive')
    target = Path('/content/drive/MyDrive/PlantScope_v4')
    target.mkdir(parents=True, exist_ok=True)
    for p in artifacts:
        if p.exists():
            shutil.copy2(p, target / p.name)
    print('Скопировано в', target)
except Exception as e:
    print('Пропущено:', e)